In [5]:
import geopandas as gpd
import xarray as xr
import numpy as np
from rasterio import features
from affine import Affine

# =========================================================
# 1. 读取 shapefile
# =========================================================
shapefile_path = (
    "/home/users/kastanie/Global_regional_analysis/"
    "atmos-WRAF01-v4-1/0.5Million KM2/"
    "region_fx-WRAF05-v4-1_WRAF_All-Hist_est1_v4-1_4-1-0_000000-000000.shp"
)

gdf = gpd.read_file(shapefile_path)

# 如果没有 region_id 字段，则自动生成
if "region_id" not in gdf.columns:
    gdf["region_id"] = np.arange(len(gdf))

# =========================================================
# 2. 读取 population nc 文件中的经纬度
# =========================================================
pop_ds = xr.open_dataset("population_1deg_2025.nc")

lon = pop_ds["lon"].values
lat = pop_ds["lat"].values

nx = len(lon)
ny = len(lat)

# =========================================================
# 3. 构建 rasterize 所需 transform
# =========================================================
# 假设为规则 1° 网格
res_lon = float(np.abs(lon[1] - lon[0]))
res_lat = float(np.abs(lat[1] - lat[0]))

# rasterio transform:
# 左上角像元边界位置
transform = Affine.translation(
    lon.min() - res_lon / 2,
    lat.min() - res_lat / 2
) * Affine.scale(res_lon, res_lat)

# =========================================================
# 4. 栅格化
# =========================================================
shapes = (
    (geom, value)
    for geom, value in zip(gdf.geometry, gdf["region_id"])
)

raster = features.rasterize(
    shapes=shapes,
    out_shape=(ny, nx),
    transform=transform,
    fill=np.nan,
    dtype="float32"
)

# =========================================================
# 5. 转换为 xarray Dataset
# =========================================================
ds = xr.Dataset(
    {
        "region_id": (
            ("lat", "lon"),
            raster
        )
    },
    coords={
        "lat": lat,
        "lon": lon
    }
)

# =========================================================
# 6. 保存为 NetCDF
# =========================================================
output_file = "region_mask_1deg.nc"

ds.to_netcdf(output_file)

print(f"✅ 保存成功: {output_file}")
print(ds)

✅ 保存成功: region_mask_1deg.nc
<xarray.Dataset> Size: 261kB
Dimensions:    (lat: 178, lon: 360)
Coordinates:
  * lat        (lat) float64 1kB -88.6 -87.6 -86.6 -85.6 ... 85.4 86.4 87.4 88.4
  * lon        (lon) float64 3kB -179.5 -178.5 -177.5 ... 177.5 178.5 179.5
Data variables:
    region_id  (lat, lon) float32 256kB 178.0 178.0 178.0 178.0 ... nan nan nan


In [4]:
import xarray as xr

ds = xr.open_dataset("population_1deg_2025.nc")

print(ds)

<xarray.Dataset> Size: 517kB
Dimensions:     (lon: 360, lat: 178)
Coordinates:
    band        int64 8B ...
  * lon         (lon) float64 3kB -179.5 -178.5 -177.5 ... 177.5 178.5 179.5
  * lat         (lat) float64 1kB -88.6 -87.6 -86.6 -85.6 ... 86.4 87.4 88.4
Data variables:
    population  (lat, lon) float64 513kB ...


In [3]:
import xarray as xr
import numpy as np
import pandas as pd

# =========================================================
# 1. 输入文件
# =========================================================
mask_file = "region_mask_1deg.nc"

population_files = {
    1985: "population_1deg_1985.nc",
    1995: "population_1deg_1995.nc",
    2005: "population_1deg_2005.nc",
    2015: "population_1deg_2015.nc",
    2020: "population_1deg_2020.nc",
    2025: "population_1deg_2025.nc",
}

output_file = "regional_population_timeseries.nc"

# =========================================================
# 2. 读取区域 mask
# =========================================================
mask_ds = xr.open_dataset(mask_file)

region_mask = mask_ds["region_id"].values

# 转整数
valid_mask = ~np.isnan(region_mask)

region_ids = np.unique(region_mask[valid_mask]).astype(int)

print(f"Number of regions: {len(region_ids)}")

# =========================================================
# 3. 创建输出数组
# =========================================================
years = list(population_files.keys())

population_out = np.full(
    (len(years), len(region_ids)),
    np.nan,
    dtype=np.float64
)

# =========================================================
# 4. 循环每个年份
# =========================================================
for yi, year in enumerate(years):

    print(f"Processing {year} ...")

    # -----------------------------------------------------
    # 读取人口数据
    # -----------------------------------------------------
    pop_ds = xr.open_dataset(population_files[year])

    pop = pop_ds["population"].values

    # -----------------------------------------------------
    # 对每个区域求和
    # -----------------------------------------------------
    for ri, rid in enumerate(region_ids):

        region_pixels = (region_mask == rid)

        total_pop = np.nansum(pop[region_pixels])

        population_out[yi, ri] = total_pop

# =========================================================
# 5. 构建 Dataset
# =========================================================
ds_out = xr.Dataset(
    {
        "population": (
            ("year", "region_id"),
            population_out
        )
    },
    coords={
        "year": years,
        "region_id": region_ids
    }
)

# =========================================================
# 6. 添加属性
# =========================================================
ds_out["population"].attrs = {
    "long_name": "Regional total population",
    "units": "people"
}

# =========================================================
# 7. 保存
# =========================================================
ds_out.to_netcdf(output_file)

print("\nSaved:")
print(output_file)

print(ds_out)

Number of regions: 237
Processing 1985 ...
Processing 1995 ...
Processing 2005 ...
Processing 2015 ...
Processing 2020 ...
Processing 2025 ...

Saved:
regional_population_timeseries.nc
<xarray.Dataset> Size: 13kB
Dimensions:     (year: 6, region_id: 237)
Coordinates:
  * year        (year) int64 48B 1985 1995 2005 2015 2020 2025
  * region_id   (region_id) int64 2kB 0 1 2 3 4 5 6 ... 231 232 233 234 235 236
Data variables:
    population  (year, region_id) float64 11kB 2.196e+04 8.379e+03 ... 9.216e+06
